## Dataset

In [ ]:
# Torchvision, PyTorch ekosisteminde görüntü verileriyle çalışmak için kullanılan bir kütüphanedir.
# datasets, MNIST ve CIFAR10 gibi hazır veri setlerine erişim sağlar.
# transforms, veriyi modele uygun forma dönüştürmek için kullanılır.
# DataLoader ise veriyi batch'ler halinde modele beslememizi sağlar.
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),  # Görüntüyü tensöre çevirir
    transforms.Normalize((0.5, 0.5, 0.5),  # RGB için 3 değer
                         (0.5, 0.5, 0.5))
])

train_data = datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
train_loader = DataLoader(train_data, batch_size=64, shuffle=True)

test_data = datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)
test_loader = DataLoader(test_data, batch_size=64, shuffle=False)

In [ ]:
print("nums of train:", len(train_data))
print("nums of test:", len(test_data))

In [ ]:
image, label = train_data[12]
print("Shape of img:", image.shape)  # örn: torch.Size([3, 32, 32])
print("Label:", label)
print("Type:", type(image))

In [ ]:
classes = ['airplane','automobile','bird','cat','deer',
           'dog','frog','horse','ship','truck']

print(classes[label])

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

img = image.permute(1, 2, 0)  # (C,H,W) → (H,W,C)
plt.imshow(img)
plt.title(classes[label])
plt.show()

In [ ]:
print(image.size(0))
print(image.size(1))
print(image.size(2))

## Model (LeNet-5)

In [ ]:
# Reference: https://github.com/activatedgeek/LeNet-5

import torch.nn as nn  # Katmanlar, aktivasyon fonksiyonları, loss fonksiyonları.
from collections import OrderedDict  # Katmanları daha düzenli tanımlamak için kullanılır.

class C1(nn.Module):  # İlk evrişim + aktivasyon + havuzlama bloğu
    def __init__(self):
        super(C1, self).__init__()

        self.c1 = nn.Sequential(OrderedDict([
            # DEĞİŞTİ: MNIST'te giriş kanalı 1 idi, CIFAR-10 RGB olduğu için 3 yaptık.
            ('c1', nn.Conv2d(in_channels=3, out_channels=6, kernel_size=(5, 5))),
            ('relu1', nn.ReLU()),
            ('s1', nn.MaxPool2d(kernel_size=(2, 2), stride=2))
        ]))

    def forward(self, img):
        output = self.c1(img)
        return output


class C2(nn.Module):  # İkinci evrişim + aktivasyon + havuzlama bloğu
    def __init__(self):
        super(C2, self).__init__()

        self.c2 = nn.Sequential(OrderedDict([
            # Bu katmanda giriş artık ilk bloktan gelen 6 feature map'tir.
            ('c2', nn.Conv2d(in_channels=6, out_channels=16, kernel_size=(5, 5))),
            ('relu2', nn.ReLU()),
            ('s2', nn.MaxPool2d(kernel_size=(2, 2), stride=2))
        ]))

    def forward(self, img):
        output = self.c2(img)
        return output


class C3(nn.Module):  # Üçüncü evrişim bloğu
    def __init__(self):
        super(C3, self).__init__()

        self.c3 = nn.Sequential(OrderedDict([
            # C2 çıkışı 16 kanallıdır, burada 120 feature map üretiyoruz.
            ('c3', nn.Conv2d(in_channels=16, out_channels=120, kernel_size=(5, 5))),
            ('relu3', nn.ReLU())
        ]))

    def forward(self, img):
        output = self.c3(img)
        return output


class F4(nn.Module):  # Tam bağlı katman
    def __init__(self):
        super(F4, self).__init__()

        self.f4 = nn.Sequential(OrderedDict([
            # 32x32 CIFAR-10 görüntüsü bu mimariden geçince C3 sonrası 120x1x1 olur.
            # Bu nedenle flatten sonrası giriş boyutu 120'dir.
            ('f4', nn.Linear(in_features=120, out_features=84)),
            ('relu4', nn.ReLU())
        ]))

    def forward(self, img):
        output = self.f4(img)
        return output


class F5(nn.Module):  # Çıkış katmanı
    def __init__(self):
        super(F5, self).__init__()

        self.f5 = nn.Sequential(OrderedDict([
            ('f5', nn.Linear(in_features=84, out_features=10)),  # CIFAR-10 -> 10 sınıf
            ('sig5', nn.LogSoftmax(dim=-1))
        ]))

    def forward(self, img):
        output = self.f5(img)
        return output


class LeNet5(nn.Module):
    """
    Input  - 3x32x32   # DEĞİŞTİ: CIFAR-10 RGB olduğu için giriş artık 3 kanallı
    Output - 10
    """
    def __init__(self):
        super(LeNet5, self).__init__()

        self.c1 = C1()
        self.c2_1 = C2()
        self.c2_2 = C2()
        self.c3 = C3()
        self.f4 = F4()
        self.f5 = F5()

    def forward(self, img):
        output = self.c1(img)   # [batch, 3, 32, 32] -> [batch, 6, 14, 14]

        x = self.c2_1(output)   # [batch, 6, 14, 14] -> [batch, 16, 5, 5]
        output = self.c2_2(output)  # Aynı girişten ikinci bir C2 bloğu geçiriliyor

        output += x  # İki C2 çıktısı toplanıyor

        output = self.c3(output)  # [batch, 16, 5, 5] -> [batch, 120, 1, 1]
        output = output.view(img.size(0), -1)  # Flatten: [batch, 120]
        output = self.f4(output)  # [batch, 84]
        output = self.f5(output)  # [batch, 10]
        return output

In [ ]:
import torch
x2 = torch.randn(2, 3, 4, 5) # Batch size, Channel, Heigth, Width
x2

In [ ]:
import torch

# Katmanları oluştur
c1 = C1()
c2_1 = C2()
c2_2 = C2()
c3 = C3()
f4 = F4()   # EKLENDİ: İlk tam bağlı katman
f5 = F5()   # EKLENDİ: Çıkış katmanı

# Input (batch=64)
x = torch.randn(64, 3, 32, 32)   # CIFAR-10 için giriş: [batch, channel, height, width]

print("Input:", x.shape)

# C1
x = c1(x)
print("After C1:", x.shape)   # Beklenen: [64, 6, 14, 14]

# C2 (iki paralel blok)
x1 = c2_1(x)
print("After C2_1:", x1.shape)   # Beklenen: [64, 16, 5, 5]

x2 = c2_2(x)
print("After C2_2:", x2.shape)   # Beklenen: [64, 16, 5, 5]

# Residual benzeri toplama
x = x1 + x2
print("After residual sum:", x.shape)   # Beklenen: [64, 16, 5, 5]

# C3
x = c3(x)
print("After C3:", x.shape)   # Beklenen: [64, 120, 1, 1]

# Flatten
x = x.view(x.size(0), -1)
print("After flatten:", x.shape)   # Beklenen: [64, 120]

# F4
x = f4(x)
print("After F4:", x.shape)   # Beklenen: [64, 84]

# F5
x = f5(x)
print("After F5:", x.shape)   # Beklenen: [64, 10]

In [ ]:
x2

## Model Train on Train Dataset

In [ ]:
import torch
import torch.optim as optim
model = LeNet5()
model2 = LeNet5()

In [ ]:
for name, param in model.named_parameters():
    print(f"Parametre adı: {name}")
    print(f"Şekli: {param.shape}")
    print(f"Değerler:\n{param}")
    print("-----------")

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)
for epoch in range(10):
    running_loss = 0.0
    for images, labels in train_loader:
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    print(f"Epoch {epoch+1}, Loss: {running_loss / len(train_loader):.4f}")

## Model Test on Test Dataset

In [ ]:
model.eval()

correct = 0
total = 0

with torch.no_grad():  
    for images, labels in test_loader:
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total
print(f'Test Accuracy: {accuracy:.2f}%')

In [ ]:
model2.eval()

correct = 0
total = 0

with torch.no_grad():  
    for images, labels in test_loader:
        outputs = model2(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total
print(f'Test Accuracy: {accuracy:.2f}%')

## ResNet18 Model

In [ ]:
#...

In [ ]:
#..

In [ ]:
# Test girdisi
x = torch.randn(64, 3, 32, 32).to(device)

print("Input:", x.shape)

# İlk katmanlar
x = model.conv1(x)
print("After conv1:", x.shape)

x = model.bn1(x)
print("After bn1:", x.shape)

x = model.relu(x)
print("After relu:", x.shape)

x = model.maxpool(x)
print("After maxpool:", x.shape)

# Residual katman grupları
x = model.layer1(x)
print("After layer1:", x.shape)

x = model.layer2(x)
print("After layer2:", x.shape)

x = model.layer3(x)
print("After layer3:", x.shape)

x = model.layer4(x)
print("After layer4:", x.shape)

# Son kısım
x = model.avgpool(x)
print("After avgpool:", x.shape)

x = torch.flatten(x, 1)
print("After flatten:", x.shape)

x = model.fc(x)
print("After fc:", x.shape)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

In [ ]:
print(model)

In [ ]:
import torch

x = torch.randn(64, 3, 32, 32).to(device)
y = model(x)

print("Output shape:", y.shape)

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)  # Learning rate 0.001 olarak belirlendi.

for epoch in range(10):
    model.train()  
    running_loss = 0.0
    
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)  
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
    
    print(f"Epoch {epoch+1}, Loss: {running_loss / len(train_loader):.4f}")

In [ ]:
model.eval()

correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total
print(f'Test Accuracy: {accuracy:.2f}%')